Banking Domain - Chain of Thought, FMA, and Prompting Patterns

This notebook reflects how I think about prompting in practice: not as a checklist of tricks, but as a way of shaping how the model reasons, responds, and stress-tests its own output.

| Pattern | My view of it | When it matters |
|---------|---------------|----------------|
| **Chain of Thought (CoT)** | Makes the model show its reasoning path before committing to an answer | High-stakes decisions with multiple variables |
| **Failure Mode Analysis (FMA)** | Pushes the model to challenge the decision by asking what could go wrong | Risk review, system design, operational planning |
| **Zero-shot** | Direct instruction with no examples | Simple classification and clear tasks |
| **One-shot** | One example teaches format and expected style | Structured output with limited examples |
| **Few-shot** | A small set of examples anchors behavior and edge cases | Triage, extraction, routing, JSON outputs |

---


In [1]:
%uv pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Users\Raunak Bansal\Desktop\venv\Scripts\python.exe: No module named uv


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

In [5]:
def chat(messages : list[dict], model="deepseek-flash"):
    """
    Function to chat with the model using the provided messages and model name.

    Args:
        messages (list[dict]): A list of dictionaries containing the chat messages.
        model (str): The name of the model to use for chatting. Default is "deepseek-flash".

    Returns:
        str: The response from the model.
    """
    response = client.chat.completions.create(
    model="deepseek-flash",
    messages=messages
)

    return response.choices[0].message.content

print("✅ Deepseek Chatbot is ready")

✅ Deepseek Chatbot is ready


---
# My understanding of Chain of Thought

To me, CoT is not about making the model sound verbose. It is about making the reasoning process explicit enough to catch weak assumptions before the final answer is produced.

In a banking context, this matters when the model has to combine facts such as affordability, risk, policy limits, and behavior. A final decision is usually better when the model is forced to examine each factor one by one before concluding.

I see CoT as a decision-quality tool: it helps the model reveal trade-offs, pressure-test logic, and reduce the chance of a shallow answer.

---
### CoT example ? Home loan eligibility review


In [6]:
cot_loan_messages = [
    {
        "role": "system",
        "content": (
            "You are a senior home loan underwriter at a leading Indian bank. "
            "When evaluating applications, always think step by step — "
            "analyse each factor individually before forming a final recommendation. "
            "Show your complete reasoning chain before the conclusion."
        )
    },
    {
        "role": "user",
        "content": """
Evaluate this home loan application step by step:

Applicant   : Ramesh Nair
Age         : 42 years
Loan Amount : ₹45,00,000
Tenure      : 20 years
Property    : ₹60,00,000 (ready possession)

Financial Profile:
  Monthly Gross Income   : ₹1,40,000
  Monthly Take-Home      : ₹1,05,000
  Existing EMIs          : ₹28,000 (car loan + personal loan)
  Credit Score           : 755
  Employment             : Salaried — Senior Manager, Wipro (11 years)
  Savings / Investments  : ₹8,00,000 in FD; ₹3,50,000 in mutual funds
  Previous Defaults      : None

Think step by step through: LTV ratio, FOIR, credit score, employment stability,
age vs tenure, and overall risk. Then give your final recommendation.
"""
    }
]

result = chat(cot_loan_messages)
print("🧠 CoT — Home Loan Eligibility:\n")
print(result)

🧠 CoT — Home Loan Eligibility:

**Applicant:** Ramesh Nair  
**Loan:** ₹45,00,000 | **Tenure:** 20 years | **Property:** ₹60,00,000 (ready possession)

---

## 1. LTV Ratio & Down Payment

- Property value = ₹60,00,000  
- Loan sought = ₹45,00,000  
- **LTV = 45,00,000 / 60,00,000 = 75%**  
- RBI norm for properties in ₹30–75 lakh bracket: max LTV generally **80%**.  
- So **75% LTV is acceptable** from a security perspective.

**However:**  
Own contribution required = ₹15,00,000 (25%).  
Applicant’s liquid savings: ₹8,00,000 FD + ₹3,50,000 MF = **₹11,50,000**.  
Shortfall for down payment alone = **₹3,50,000**, and after stamp duty/registration/processing fees, the gap is likely **₹6–8 lakh or more**.  
➡ **Source of own contribution needs verification.**

---

## 2. EMI & FOIR

Assume interest rate around **9% p.a.** for calculation.

- EMI for ₹45,00,000 for 20 years @ 9% ≈ **₹40,488/month**
- Existing EMIs = **₹28,000/month**
- **Total EMIs = ₹68,488/month**

**FOIR calculations:*

---
### 🔹 CoT Use Case 2 — Real-Time Transaction Fraud Scoring

In [7]:
cot_fraud_messages = [
    {
        "role": "system",
        "content": (
            "You are a real-time fraud detection analyst at a bank. "
            "For every transaction flagged for review, reason through each risk signal "
            "one at a time before assigning a fraud risk score (0-100) and a decision: "
            "Allow / Flag for Review / Block."
        )
    },
    {
        "role": "user",
        "content": """
Analyse this transaction for fraud. Think step by step.

Transaction Details:
  Card Holder     : Anita Verma (account since 2017)
  Transaction Time: 02:47 AM IST
  Amount          : ₹92,500
  Merchant        : Electronics store, Frankfurt, Germany
  Card Type       : Domestic Debit Card
  Last Transaction: ₹350 at Chennai supermarket, 6 hours ago
  International Tx: Never enabled on this card
  Device          : New device fingerprint (not seen before)
  OTP             : Entered correctly

Reason through each anomaly signal individually, assign a fraud risk score, and give your final decision.
"""
    }
]

result = chat(cot_fraud_messages)
print("🧠 CoT — Fraud Transaction Scoring:\n")
print(result)

🧠 CoT — Fraud Transaction Scoring:

## Step-by-step fraud risk analysis

| Signal | Assessment | Risk |
|---|---|---|
| **Cardholder tenure** | Anita Verma has been an account holder since 2017. Long tenure is generally a positive/trust signal, but it does not override transaction-level anomalies. | Low |
| **Transaction time** | 02:47 AM IST is an unusual hour for a ₹92,500 electronics purchase. Combined with the last transaction only 6 hours earlier, this does not fit normal spending behavior. | Medium-High |
| **Amount** | ₹92,500 is very high compared with the last transaction of ₹350. That is roughly 264× the prior amount. Electronics are also high-resale-value goods commonly targeted by fraudsters. | High |
| **Merchant & location** | Electronics store in Frankfurt, Germany. International high-value electronics transactions are a common fraud pattern, especially when the card is domestic and has never had international usage enabled. | High |
| **Card type / international setting

---
### 🔹 CoT Use Case 3 — AML (Anti-Money Laundering) Suspicious Activity Report

In [10]:
cot_aml_messages = [
    {
        "role": "system",
        "content": (
            "You are a compliance officer specialising in Anti-Money Laundering (AML) at a commercial bank. "
            "When reviewing customer activity, reason through each AML red flag step by step — "
            "structuring, layering, placement, velocity, counterparty risk — "
            "before deciding whether to file a Suspicious Activity Report (SAR)."
        )
    },
    {
        "role": "user",
        "content": """
Review this customer's activity for AML red flags. Think step by step.

Customer    : Vikram Sood, Sole Proprietor — Import/Export firm
Account Age : 14 months
KYC Status  : Complete

Activity Over Last 30 Days:
  - 18 cash deposits ranging ₹45,000–₹49,500 each (total ₹8,64,000)
  - 3 large RTGS outward transfers to Dubai (₹2,10,000 / ₹1,95,000 / ₹2,30,000)
  - Multiple round-number transfers to 6 different new beneficiaries
  - No corresponding inward trade-related payments or invoices on record
  - Average monthly turnover before this period: ₹1,20,000

Reason through each AML red flag, score the suspicion level (Low/Medium/High/Critical),
and recommend: No Action / Enhanced Monitoring / File SAR.
"""
    }
]

result = chat(cot_aml_messages)
print("🧠 CoT — AML Suspicious Activity Analysis:\n")
print(result)

🧠 CoT — AML Suspicious Activity Analysis:

**Overall suspicion level: Critical**  
**Recommendation: File SAR/STR with FIU-IND immediately** — not just Enhanced Monitoring.

### Step-by-step AML red flag review

| # | Red flag | AML concern | Suspicion |
|---|---|---|---|
| 1 | **18 cash deposits of ₹45,000–₹49,500 each** | Classic **structuring/placement**: amounts sit just below the ₹50,000 threshold commonly linked to PAN/record-keeping and internal reporting triggers. Total ₹8.64L over 30 days. | **Critical** |
| 2 | **Activity vs. historical turnover** | Average monthly turnover was ₹1.20L; now ₹8.64L cash deposits alone in 30 days — roughly **7x normal velocity**. Sudden unexplained change. | **High** |
| 3 | **Import/export sole proprietor making large cash deposits** | Import/export trade normally flows through banking channels, invoices, LCs, customs documents, bills of lading, etc. Cash-intensive behaviour is inconsistent with the stated business model. | **High** |
| 4 | **3

---
# My understanding of Failure Mode Analysis

FMA is the opposite lens of CoT. Instead of asking, "What is the answer?", it asks, "What can break this answer, why would it fail, and what controls are needed?"

For me, this is the right pattern when the problem is not just predictive accuracy, but operational resilience. In banking, the real risk is often not a bad model output alone; it is a bad model output combined with weak controls, poor monitoring, or an overlooked failure path.

FMA helps translate a decision into a risk review. It turns a single recommendation into a system-level conversation about exposure, controls, and recovery.

---
### FMA example ? Core banking system migration


In [11]:
fma_cbs_messages = [
    {
        "role": "system",
        "content": (
            "You are a banking IT risk consultant specialising in core banking transformations. "
            "When asked to perform Failure Mode Analysis, identify every significant failure mode, "
            "its root cause, probability (High/Medium/Low), business impact (Critical/High/Medium/Low), "
            "and a specific mitigation strategy. Structure your output as a clear numbered list."
        )
    },
    {
        "role": "user",
        "content": """
Perform a Failure Mode Analysis for the following banking scenario:

Scenario: A mid-sized private bank (2.4 million customers) is migrating its Core Banking System 
from a legacy on-premise platform to a new cloud-based CBS over a single weekend cutover window.

Scope includes: savings & current accounts, fixed deposits, retail loans, NEFT/RTGS/UPI, 
debit card issuance, and internet banking portal.

Identify all failure modes — technical, operational, data, compliance, and customer-facing.
For each, provide: root cause, probability, business impact, and mitigation.
"""
    }
]

result = chat(fma_cbs_messages)
print("⚠️ FMA — Core Banking Migration:\n")
print(result)

⚠️ FMA — Core Banking Migration:

## Failure Mode Analysis — Single-Weekend Core Banking System Migration

**Assumptions:** Big-bang cutover over one weekend; legacy system kept warm/rollback-capable unless stated; scope includes savings/current accounts, fixed deposits, retail loans, NEFT/RTGS/UPI, debit card issuance, and internet banking. Ratings are **pre-mitigation** and indicative.

### Technical & Infrastructure

1. **Under-provisioned cloud capacity / performance degradation** — **Root cause:** Sizing based on average load, not month-end/salary-day peaks; cold-start autoscaling; noisy-neighbour cloud tenancy. **Probability:** High. **Business impact:** Critical. **Mitigation:** Capacity-test at 2–3× peak TPS; pre-warm autoscaling; reserve instances; secure cloud-provider enterprise SLA; run 24/7 war room during cutover.

2. **Cloud region/AZ outage or provider service failure** — **Root cause:** Single-region or single-AZ deployment; dependency on one provider service. **Probab

---
### FMA Use Case 2 — Credit Card Instant Approval Workflow

In [12]:
fma_cc_messages = [
    {
        "role": "system",
        "content": (
            "You are a credit risk and operations manager at a bank. "
            "Perform rigorous Failure Mode Analysis on banking processes. "
            "For each failure mode, provide: failure description, trigger/root cause, "
            "probability (High/Medium/Low), financial/reputational impact, "
            "detection method, and recommended control."
        )
    },
    {
        "role": "user",
        "content": """
Perform a Failure Mode Analysis on this process:

Process: Instant Credit Card Approval — an AI-driven workflow that takes a customer 
application (via mobile app), pulls bureau data, runs an internal credit model, 
and issues a virtual card number within 60 seconds if approved.

Analyse failure modes across these dimensions:
1. Data & Bureau Integration failures
2. AI Model failures
3. Identity & KYC verification failures
4. Fraud & synthetic identity risks
5. Regulatory & compliance failures
6. Customer experience failures

Be specific to the banking and credit card context.
"""
    }
]

result = chat(fma_cc_messages)
print("⚠️ FMA — Credit Card Instant Approval Workflow:\n")
print(result)

⚠️ FMA — Credit Card Instant Approval Workflow:

## FMEA: Instant Credit Card Approval — AI-Driven Mobile Workflow

**Scope assumption:** U.S. consumer credit card issuer; application via mobile app; bureau pull; internal credit model; virtual card issued within 60 seconds if approved. Regulated by FCRA, Reg B/ECOA, TILA/Reg Z, CARD Act, BSA/AML, OFAC, GLBA, PCI DSS, UDAAP, and SR 11-7 model risk expectations.

**Rating key**  
- **Prob.:** H = likely in normal operations; M = possible; L = unlikely but plausible.  
- **Impact (Fin/Rep):** first = financial, second = reputational. H = material loss/regulatory action/major reputational harm; M = moderate; L = low.  
- Ratings are **inherent risk before controls** and should be recalibrated with institution-specific data.

---

### 1. Data & Bureau Integration Failures

| Failure mode & description | Trigger / root cause | Prob. | Impact (Fin/Rep) | Detection | Recommended control |
|---|---|---|---|---|---|
| Bureau API outage/timeout s

---
### 🔸 FMA Use Case 3 — UPI Payment Failure Root Cause Analysis

In [13]:
fma_upi_messages = [
    {
        "role": "system",
        "content": (
            "You are a payments technology expert at a bank's digital banking division. "
            "When asked to analyse payment failures, identify all failure modes with "
            "layered root cause analysis (immediate cause → contributing cause → root cause), "
            "customer impact, RBI/NPCI compliance implications, and resolution playbook steps."
        )
    },
    {
        "role": "user",
        "content": """
Perform a Failure Mode Analysis on UPI payment failures at our bank.

Context: Our bank processes ~8 lakh UPI transactions per day. 
Over the past month, we observed:
  - 3.2% transaction failure rate (industry benchmark: <0.5%)
  - Spike in failures between 6–9 PM daily
  - Highest failures on salary credit days (1st and 7th of month)
  - Error codes seen: U16 (bank temporarily unavailable), U30 (transaction declined by bank), 
    U68 (request timed out)
  - Customer complaints up 240% in the same period

Analyse all failure modes — infrastructure, switch, CBS integration, NPCI interface, 
and peak load handling. Provide a layered root cause analysis and resolution playbook.
"""
    }
]

result = chat(fma_upi_messages)
print("⚠️ FMA — UPI Payment Failures:\n")
print(result)

⚠️ FMA — UPI Payment Failures:

# UPI Failure Mode Analysis & Resolution Playbook

## 0. Baseline & Assumptions

| Metric | Value |
|---|---:|
| Daily UPI volume | ~8,00,000 |
| Observed failure rate | 3.2% = ~25,600 failures/day |
| Benchmark | <0.5% = <4,000 failures/day |
| Excess failures | ~21,600/day; ~6.48 lakh/month |
| Peak window | 6–9 PM daily |
| Spike days | 1st and 7th (salary credit days) |
| Error codes | U16, U30, U68 |
| Complaints | +240% |

**Average TPS:** 8,00,000 ÷ 86,400 ≈ **9.3 TPS**  
**6–9 PM peak:** if 40% of daily volume occurs in 3 hours → ~3.2 lakh txns/3h ≈ **30 TPS average**, with micro-peaks likely **60–90 TPS**.  
**Salary days:** volume may double → peak could reach **120–180 TPS**.  

The correlation with 6–9 PM and salary days strongly indicates **capacity/saturation failures**, not random business declines.

---

## 1. Failure Mode Analysis

### 1.1 Infrastructure Failures

| Failure Mode | Error Codes | Layered RCA | Customer / Compliance Impact 

---
# My thinking on combining CoT + FMA

This is the pattern I find most useful in real decision-making: first reason through the case, then stress-test that reasoning against what could go wrong.

CoT gives the model a structured path to the answer. FMA gives it a second pass to challenge that answer. Together, they create a much more realistic decision process than either one alone.

This is especially valuable in banking, where a good decision is not just correct in theory; it must survive operational, regulatory, and customer-risk contexts.

---
### Example ? Digital lending product launch


In [14]:
combo_messages = [
    {
        "role": "system",
        "content": (
            "You are a chief risk officer (CRO) at a digital bank. "
            "When evaluating new product proposals: "
            "FIRST use Chain of Thought to reason through the business case step by step, "
            "THEN perform a Failure Mode Analysis to identify risks that could derail the product. "
            "Clearly label the two sections: [CHAIN OF THOUGHT] and [FAILURE MODE ANALYSIS]."
        )
    },
    {
        "role": "user",
        "content": """
Our bank wants to launch 'SalaryAdvance' — an instant, zero-collateral micro-loan product:

Product Details:
  - Target customers  : Salaried individuals with salary account at our bank (min 3 months)
  - Loan amount       : ₹5,000 to ₹50,000
  - Tenure            : 1–3 months
  - Interest rate     : 1.5% per month (flat)
  - Approval          : Fully automated, decision in under 90 seconds
  - Repayment         : Auto-debit on next salary credit date
  - No documentation  : Approved purely on salary history + internal credit score

STEP 1 — Think step by step through: market opportunity, credit risk, 
         revenue potential, regulatory fit, and operational readiness.

STEP 2 — Then perform a Failure Mode Analysis: what are all the ways 
         this product could fail financially, operationally, or reputationally?
"""
    }
]

result = chat(combo_messages)
print("🔗 CoT + FMA — SalaryAdvance Product Launch:\n")
print(result)

🔗 CoT + FMA — SalaryAdvance Product Launch:

[CHAIN OF THOUGHT]

As CRO, I would evaluate this as a risk-adjusted return question: can SalaryAdvance earn enough after expected credit loss, fraud, servicing, funding, capital, and compliance costs?

1. **Market opportunity**
   - Salaried customers with salary accounts at our bank are attractive: known cash flow, lower acquisition cost, auto-debit potential, cross-sell opportunity.
   - Real demand exists for short-term liquidity before salary, but competitors include credit cards, overdrafts, BNPL, personal loans, and salary-advance apps.
   - Minimum 3 months salary history is weak for credit assessment. It may attract thin-file or highly leveraged customers.
   - Opportunity is real, but only if underwriting is stronger than “salary credit + internal score.”

2. **Credit risk**
   - Unsecured, no collateral, fully automated, thin-file lending. Key risks: job loss, salary delay, job switch, over-indebtedness, fraud, model error.
   - S

---
## My mental model of prompting patterns

I think of prompting as a spectrum of guidance. The more structure I give the model, the more I constrain its behavior - but I also increase the chance of getting a consistent, task-specific response.

| Pattern | What it does in plain terms | My use case |
|---------|----------------------------|-------------|
| **Zero-shot** | Directly asks for the result without examples | Fast classification, direct answer generation, simple policy checks |
| **One-shot** | Gives one example to teach the format or tone | Consistent output structure, templated messages, simple transformations |
| **Few-shot** | Gives a few examples so the model learns patterns and edge cases | Triage, routing, extraction, risk tagging, classification with nuance |
| **CoT** | Makes reasoning visible before the final answer | Multi-factor decisions, underwriting, fraud review, AML investigations |
| **FMA** | Challenges the plan by surfacing failure modes and controls | Risk reviews, product launch assessment, operational resilience analysis |

### Why this matters to me

- **Zero-shot** is the cleanest and fastest approach when the task is simple and well-specified.
- **One-shot** is useful when I want the model to learn the expected format without overfitting to too many examples.
- **Few-shot** is stronger when I need pattern recognition, similarity, and consistency across edge cases.
- **CoT** is valuable when the decision depends on multiple steps, assumptions, or trade-offs.
- **FMA** is essential when the task is not just "decide" but "decide safely under uncertainty."

This is how I think about prompting: the goal is not to force a format for its own sake, but to choose the right level of guidance for the decision being made.
